In [1]:
%pylab inline
import gc
import fitsio
import treecorr
import numpy as np
import catutil
import matplotlib.lines as mlines

import astropy.table as astTable

Populating the interactive namespace from numpy and matplotlib


In [ ]:
nzs    =   4
pzname =   'dnnz' #demp/mizuki/dnnz
dataG  =   [[],[],[],[]]
pzDir   =   'wrkDir/catalog/'
for fieldname in catutil.field_names:
    # mock catalog (will copy v2 to GW and idark)
    catFname=  'wrkDir/g3Sim/catalog_obs_reGaus_no_m/%s_no_m.fits' %(fieldname)
    data   =   fitsio.read(catFname)[['i_ra','i_dec','i_sdssshape_psf_shape11',\
                                    'i_sdssshape_psf_shape12','i_sdssshape_psf_shape22']]
    pzsname=   os.path.join(pzDir,'%s_%s_zmask.fits' %(fieldname,pzname))
    for iz in range(nzs):
        zmsk   =   fitsio.read(pzsname)['%s_mask_%sbin' %(pzname,iz)]
        # merge observational fields
        dataG[iz].append(data[zmsk])
    del data
    gc.collect()
    break
dataG=[np.hstack(dataG[i]) for i in range(nzs)]

In [24]:
cor  =   treecorr.NKCorrelation(nbins=30,min_sep=0.281838293,max_sep=354.8133892,sep_units='arcmin')
out  =   []
for ii in range(nzs):
    cat1  =   treecorr.Catalog(ra=dataG[ii]['i_ra'],dec=dataG[ii]['i_dec'],ra_units='deg',dec_units='deg') 
    cat2  =   treecorr.Catalog(k=catutil.get_psf_size(dataG[ii]),
                                ra=dataG[ii]['i_ra'],dec=dataG[ii]['i_dec'],ra_units='deg',dec_units='deg') 
    # correlation 
    cor.clear()
    cor.process(cat1,cat2)
    del cat1,cat2
    gc.collect()
    out.append(cor.copy())